## Fase 2: Análisis Morfosintáctico y Extracción de Features

---
A partir de ahora, usaremos la submuestra establecida del corpus.

---

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">2.1. ¿Cuántos registros contiene el corpus?</span>

In [22]:
registros = data_limpio.shape[0]
registros_sub = sub_data.shape[0]

print(f"El corpus original contiene {registros} registros")
print(f"La submuestra extraída del corpus original contiene {registros_sub} registros")

El corpus original contiene 574630 registros
La submuestra extraída del corpus original contiene 39998 registros


<hr>
Aunque la submuestra extraída del corpus original es considerablemente más pequeña, mantiene las proporciones de las variables que se estudiarán en las actividades. De este modo, se obtiene un conjunto de datos representativo del corpus original, lo que permite reducir significativamente el gran coste computacional que supone para mi equipo emplear el dataset completo.

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">2.2. ¿Cuántas palabras totales hay en los comentarios del corpus?</span>

In [23]:
# Inicializamos la variable del conteo de palabras de los comentarios
conteo_pal_com = 0

# Inicializamos la variable del conteo de palabras totales
conteo_total = 0

# Fijamos el número de filas
num_filas = sub_data.shape[0]

# Realizamos un bucle simple para el conteo de palabras de los comentarios y el conteo de palabras totales
for i in range(num_filas):

  # Aseguramos que sea tipo string la celda
  texto = str(sub_data.loc[i,"CONTENIDO A ANALIZAR"])
  doc = nlp(texto)

  # Fijamos el conteo de palabras a 0 y contamos
  num_tokens_alpha = 0
  for token in doc:
    if token.is_alpha:
      num_tokens_alpha = num_tokens_alpha + 1

  # Sumamos al conteo total
  conteo_total = conteo_total + num_tokens_alpha

  # Sumamos al conteo de palabras de comentarios si es un comentario
  if(sub_data.loc[i, "TIPO DE MENSAJE"] == "COMENTARIO"):
    conteo_pal_com = conteo_pal_com + num_tokens_alpha

print(f"En los comentarios del corpus hay un total de {conteo_pal_com} palabras de las {conteo_total} palabras totales.")
print(f"Las palabras en los comentarios suponen el {conteo_pal_com*100/conteo_total:.2f}% de las totales")

En los comentarios del corpus hay un total de 1089728 palabras de las 4212432 palabras totales.
Las palabras en los comentarios suponen el 25.87% de las totales


<hr>
La palabras en comentarios representan alrededor del 26 % de las palabras totales del contenido a analizar. Es decir, aporximadamente una cuarta parte de todas las palabras provienen de comentarios, lo cual es un dato relevante a tener en cuenta en el análisis.

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">2.3. ¿Cuál el número promedio de palabras en cada comentario?</span>

In [24]:
# Fijamos el numero de comentarios
num_comentarios = sub_data['TIPO DE MENSAJE'].value_counts()['COMENTARIO']

print(f"El promedio de palabras en cada comentario es de {round(conteo_pal_com/num_comentarios)} palabras")

El promedio de palabras en cada comentario es de 47 palabras


<hr>
Una longitud media de 47 palabras se considera suficiente para afirmar que los comentarios no constituyen textos breves. En consecuencia, la intensidad de odio detectada en cada uno de ellos resulta más representativa, ya que es menos probable que se interpreten de forma incorrecta las intenciones del autor. En comentarios más cortos podría introducirse un mayor sesgo interpretativo, al estar menos contextualizados.

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">2.4. Considerando dos grupos de comentarios (odio y no odio) ¿Cuál el número promedio de palabras en los comentarios de cada grupo?</span>

In [25]:
# Inicializamos la variable del conteo de palabras de los comentarios sin odio
conteo_sin_odio = 0

# Inicializamos la variable del conteo de palabras de los comentarios con odio
conteo_con_odio = 0

# Realizamos un bucle para el conteo de las palabras
for i in range(num_filas):

  # Aseguramos que es tipo string la celda
  texto = str(sub_data.loc[i,"CONTENIDO A ANALIZAR"])
  doc = nlp(texto)

  # Fijamos el conteo de palabras a 0 y contamos
  num_tokens_alpha = 0
  for token in doc:
    if token.is_alpha:
      num_tokens_alpha += 1

  # Sumamos al conteo correspondiente
  if(sub_data.loc[i, "TIPO DE MENSAJE"] == "COMENTARIO" and sub_data.loc[i,"INTENSIDAD"] == 0):
    conteo_sin_odio += num_tokens_alpha
  elif(sub_data.loc[i, "TIPO DE MENSAJE"] == "COMENTARIO" and sub_data.loc[i,"INTENSIDAD"] > 0):
    conteo_con_odio += num_tokens_alpha

total_sin_odio = sub_data[(sub_data["TIPO DE MENSAJE"] == "COMENTARIO") & (sub_data["INTENSIDAD"] == 0.0)].shape[0]
total_con_odio = sub_data[(sub_data["TIPO DE MENSAJE"] == "COMENTARIO") & (sub_data["INTENSIDAD"] != 0.0)].shape[0]

promedio_pal_sin = conteo_sin_odio / total_sin_odio if total_sin_odio > 0 else 0
promedio_pal_con = conteo_con_odio / total_con_odio if total_con_odio > 0 else 0

print(f"El promedio de palabras en comentarios sin odio es de {round(promedio_pal_sin, 2)} palabras")
print(f"El promedio de palabras en comentarios con odio es de {round(promedio_pal_con, 2)} palabras")

El promedio de palabras en comentarios sin odio es de 47.85 palabras
El promedio de palabras en comentarios con odio es de 15.57 palabras


<hr>
Los comentarios con odio se expresan, de media, en textos más breves. Los comentarios más extensos tienden a estar menos asociado a la presencia de odio en el corpus. Podemos deducir con estos datos, y teniendo en cuenta que el promedio de palabras sin distinción entre odio/no odio es de 47, que la gran parte de los comentarios no contienen odio.

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">2.5. Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el número promedio de oraciones en los comentarios de cada grupo?</span>

In [26]:
# Inicializamos contadores de oraciones y de comentarios por tipo de odio
conteo_orac_sin_odio = 0
conteo_orac_con_odio = 0
num_comentarios_sin = 0
num_comentarios_con = 0

# Realizamos un bucle para el conteo de las oraciones
for i in range(num_filas):

  # Aseguramos que es tipo string la celda
  texto = str(sub_data.loc[i,"CONTENIDO A ANALIZAR"])
  doc = nlp(texto)

  if(sub_data.loc[i, "TIPO DE MENSAJE"] == "COMENTARIO" and sub_data.loc[i,"INTENSIDAD"] == 0):
    conteo_orac_sin_odio += len(list(doc.sents))
    num_comentarios_sin += 1
  elif(sub_data.loc[i, "TIPO DE MENSAJE"] == "COMENTARIO" and sub_data.loc[i,"INTENSIDAD"] > 0):
    conteo_orac_con_odio += len(list(doc.sents))
    num_comentarios_con += 1

promedio_orac_sin = conteo_orac_sin_odio / num_comentarios_sin if num_comentarios_sin > 0 else 0
promedio_orac_con = conteo_orac_con_odio / num_comentarios_con if num_comentarios_con > 0 else 0

print(f"Promedio de oraciones por comentario sin odio: {promedio_orac_sin:.2f}")
print(f"Promedio de oraciones por comentario con odio: {promedio_orac_con:.2f}")

Promedio de oraciones por comentario sin odio: 2.26
Promedio de oraciones por comentario con odio: 1.51


<hr>
La diferencia observada en la conclusión anterior se presenta también en el número de oraciones de los comentarios, lo que refuerza dicha afirmación.

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">2.6. Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el porcentaje de palabras en cada combinación posible de género y número (p.ej. masculino singular) en cada grupo?</span>

In [27]:
conteo_gn_sin_odio = {
    ("Masc", "Sing"): 0,
    ("Masc", "Plur"): 0,
    ("Fem", "Sing"): 0,
    ("Fem", "Plur"): 0
}

conteo_gn_con_odio = {
    ("Masc", "Sing"): 0,
    ("Masc", "Plur"): 0,
    ("Fem", "Sing"): 0,
    ("Fem", "Plur"): 0
}

total_sin_odio = 0
total_con_odio = 0

# Número de filas 
num_filas = sub_data.shape[0]

# Bucle principal
for i in range(num_filas):

    # Solo analizamos comentarios
    if sub_data.loc[i, "TIPO DE MENSAJE"] != "COMENTARIO":
        continue

    texto = str(sub_data.loc[i, "CONTENIDO A ANALIZAR"])
    doc = nlp(texto)
    
    intensidad = sub_data.loc[i, "INTENSIDAD"]

    for ent in doc.ents:
        for token in ent:

            if token.pos_ not in {"NOUN", "PROPN", "ADJ", "DET", "PRON"}:
                continue

            genero = token.morph.get("Gender") # El get devuelve una lista
            numero = token.morph.get("Number") # El get devuelve una lista

            if genero and numero:
                clave = (genero[0], numero[0]) 

                if intensidad == 0:
                    conteo_gn_sin_odio[clave] += 1
                    total_sin_odio += 1
                elif intensidad > 0:
                    conteo_gn_con_odio[clave] += 1
                    total_con_odio += 1

# Resultados
print("Las proporciones de género y número en entidades NER de comentarios son:")

for clave, valor in conteo_gn_sin_odio.items():
    genero, numero = clave
    print(f"  {genero} {numero} en comentarios sin odio: {valor} ({valor/sum(conteo_gn_sin_odio.values())*100:.2f}%)")

print(f" Total sin odio: {total_sin_odio}")
for clave, valor in conteo_gn_con_odio.items():
    genero, numero = clave
    print(f"  {genero} {numero} en comentarios con odio: {valor} ({valor/sum(conteo_gn_con_odio.values())*100:.2f}%)")

print(f" Total con odio: {total_con_odio}")

Las proporciones de género y número en entidades NER de comentarios son:
  Masc Sing en comentarios sin odio: 3882 (43.08%)
  Masc Plur en comentarios sin odio: 812 (9.01%)
  Fem Sing en comentarios sin odio: 3769 (41.83%)
  Fem Plur en comentarios sin odio: 548 (6.08%)
 Total sin odio: 9011
  Masc Sing en comentarios con odio: 53 (35.81%)
  Masc Plur en comentarios con odio: 16 (10.81%)
  Fem Sing en comentarios con odio: 73 (49.32%)
  Fem Plur en comentarios con odio: 6 (4.05%)
 Total con odio: 148


<hr>
Los comentarios se expresan mayormente en singular; sin embargo, se puede observar que en los comentarios con odio predomina ligeramente el uso del género femenino.  